In [3]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import pandas as pd

# Set your Spotify API credentials
client_id = 'efeb5f49e3494abeb5488faad890f79d'  # Replace with your client ID
client_secret = 'f115b08e67b64c9f9b3083e20456b702'  # Replace with your client secret
redirect_uri = 'http://localhost:8888/callback'  # Redirect URI, can be any localhost URI

# Define the scope for accessing playlist data
scope = 'playlist-read-private'

# Authenticate with Spotify
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=client_id,
                                               client_secret=client_secret,
                                               redirect_uri=redirect_uri,
                                               scope=scope))

# Function to retrieve playlist data
def get_playlist_tracks(playlist_id):
    results = sp.playlist_tracks(playlist_id)
    tracks = results['items']
    while results['next']:
        results = sp.next(results)
        tracks.extend(results['items'])
    return tracks

# Replace with your playlist IDs
playlist_ids = [
    '7esVNJViQYK5JOd86VowbK',
    '4l6rfZHb9OcUfCdlB2lu3z',
    '7iXz9d9A62kAoJ84Vv8oh1'
]

# Initialize an empty DataFrame to store the playlist data
playlist_data = []

# Loop through each playlist and retrieve its data
for pid in playlist_ids:
    tracks = get_playlist_tracks(pid)
    for track in tracks:
        track_info = {
            'Track Name': track['track']['name'],
            'Artist': track['track']['artists'][0]['name'],
            'Album': track['track']['album']['name'],
            'Spotify URI': track['track']['uri'],
            'Track ID': track['track']['id'],
            'Duration (ms)': track['track']['duration_ms']
        }
        playlist_data.append(track_info)

# Convert the data into a DataFrame
playlist_df = pd.DataFrame(playlist_data)

# Save the combined playlist data to a CSV file
playlist_df.to_csv('Combined_Playlist.csv', index=False)

# Display the first few rows of the DataFrame
playlist_df.head()



,Track Name,Artist,Album,Spotify URI,Track ID,Duration (ms)
0,My Baby Just Cares for Me - 2013 Remastered Ve...,Nina Simone,Little Girl Blue (2013 Remastered Version),spotify:track:6VTbbVjKOC2qWagIDbkJrC,6VTbbVjKOC2qWagIDbkJrC,215746
1,I Put A Spell On You,Nina Simone,I Put A Spell On You,spotify:track:0sjxRg1VlYfx4YG7uxurrq,0sjxRg1VlYfx4YG7uxurrq,155026
2,Feeling Good,Nina Simone,I Put A Spell On You,spotify:track:6Rqn2GFlmvmV4w9Ala0I1e,6Rqn2GFlmvmV4w9Ala0I1e,174000
3,(You Make Me Feel Like) A Natural Woman,Aretha Franklin,Lady Soul (With Bonus Selections),spotify:track:5K09WxKdlkroDyEVyOSySy,5K09WxKdlkroDyEVyOSySy,165333
4,Chain of Fools,Aretha Franklin,Lady Soul (With Bonus Selections),spotify:track:2oZmMp5M6L0Rh7G84Um2tF,2oZmMp5M6L0Rh7G84Um2tF,167066


In [13]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import pandas as pd

# Set your Spotify API credentials
client_id = 'efeb5f49e3494abeb5488faad890f79d'  # Replace with your client ID
client_secret = 'f115b08e67b64c9f9b3083e20456b702'  # Replace with your client secret
redirect_uri = 'http://localhost:8888/callback'  

# Define the scope for accessing playlist data
scope = 'playlist-read-private'

# Authenticate with Spotify
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=client_id,
                                               client_secret=client_secret,
                                               redirect_uri=redirect_uri,
                                               scope=scope))

# Function to retrieve playlist data
def get_playlist_tracks(playlist_id):
    try:
        results = sp.playlist_tracks(playlist_id, limit=100)
        tracks = results['items']
        while results['next']:
            results = sp.next(results)
            tracks.extend(results['items'])
        return tracks
    except spotipy.exceptions.SpotifyException as e:
        print(f"SpotifyException: {e}")
        return []
    except Exception as e:
        print(f"General Exception: {e}")
        return []

# Function to get audio features for a list of track IDs
def get_audio_features(track_ids):
    try:
        features = sp.audio_features(track_ids)
        # Filter out None values and return only valid features
        return [feature for feature in features if feature is not None]
    except Exception as e:
        print(f"Error retrieving audio features: {e}")
        return []


# Replace with your extracted playlist IDs
playlist_ids = [
    '7esVNJViQYK5JOd86VowbK',
    '4l6rfZHb9OcUfCdlB2lu3z',
    '7iXz9d9A62kAoJ84Vv8oh1'
]

# Initialize an empty list to store the playlist data
playlist_data = []

# Loop through each playlist and retrieve its data
for pid in playlist_ids:
    tracks = get_playlist_tracks(pid)
    for track in tracks:
        track_info = {
            'Track Name': track['track']['name'],
            'Artist': track['track']['artists'][0]['name'],
            'Album': track['track']['album']['name'],
            'Spotify URI': track['track']['uri'],
            'Track ID': track['track']['id'],
            'Duration (ms)': track['track']['duration_ms']
        }
        playlist_data.append(track_info)

# Convert the data into a DataFrame
playlist_df = pd.DataFrame(playlist_data)

# Get audio features for all tracks in the playlist
all_audio_features = []
chunk_size = 100

for i in range(0, len(playlist_df), chunk_size):
    chunk = playlist_df['Track ID'].iloc[i:i + chunk_size].tolist()
    audio_features = get_audio_features(chunk)
    all_audio_features.extend(audio_features)

# Convert the audio features into a DataFrame
# Convert the audio features into a DataFrame
audio_features_df = pd.DataFrame(all_audio_features)

# Print audio features DataFrame for debugging
print("Audio Features DataFrame:")
print(audio_features_df.head())
print("Columns in Audio Features DataFrame:", audio_features_df.columns.tolist())


# Combine the original playlist data with the audio features data
# Reset the index to ensure proper alignment
combined_playlist = pd.concat([playlist_df.reset_index(drop=True), audio_features_df.reset_index(drop=True)], axis=1)

# Print the combined playlist DataFrame for debugging
print("Combined Playlist DataFrame:")
print(combined_playlist.head())
print("Columns in Combined Playlist DataFrame:", combined_playlist.columns.tolist())

required_columns = ['Energy', 'Acousticness', 'Danceability', 'Tempo']

# Check if required columns are present
missing_columns = [col for col in required_columns if col not in combined_playlist.columns]
if missing_columns:
    print(f"Missing columns: {missing_columns}")
else:
    print("All required columns are present.")




Audio Features DataFrame:
   danceability  energy  key  loudness  mode  speechiness  acousticness  \
0         0.746   0.243    4   -15.134     0       0.1230         0.829   
1         0.428   0.398    6    -9.130     0       0.0295         0.755   
2         0.451   0.326    7    -9.239     0       0.0372         0.863   
3         0.603   0.271    5   -10.083     1       0.0284         0.679   
4         0.721   0.567   10    -9.516     1       0.0389         0.221   

   instrumentalness  liveness  valence    tempo            type  \
0          0.000001    0.0905    0.425  118.578  audio_features   
1          0.000537    0.1720    0.365   88.886  audio_features   
2          0.000000    0.1920    0.550   81.726  audio_features   
3          0.000000    0.1410    0.405  110.890  audio_features   
4          0.000000    0.4650    0.914  116.643  audio_features   

                       id                                   uri  \
0  6VTbbVjKOC2qWagIDbkJrC  spotify:track:6VTbbVjKOC2q

In [27]:
# Initialize empty playlists
easy_listening = []
upbeat_dance = []
chill_reflective = []
unselected = []  # New playlist for unselected tracks

# Categorize tracks with refined criteria
for index, row in combined_playlist.iterrows():
    try:
        # Ensure the necessary columns are not None before accessing
        if pd.notnull(row['energy']) and pd.notnull(row['acousticness']) and pd.notnull(row['danceability']) and pd.notnull(row['tempo']):
            print(f"Processing Track: {row['Track Name']} by {row['Artist']}")
            print(f"Energy: {row['energy']}, Acousticness: {row['acousticness']}, Danceability: {row['danceability']}, Tempo: {row['tempo']}")

            # Revised criteria for categorization
            if row['energy'] < 0.55 and row['acousticness'] > 0.16 and row['danceability'] < 0.6:
                easy_listening.append(row)
                print("Added to Easy Listening Playlist")
            elif row['energy'] > 0.5 and row['danceability'] > 0.6 and row['tempo'] > 90:
                upbeat_dance.append(row)
                print("Added to Upbeat Dance Playlist")
            elif row['energy'] < 0.65 and row['acousticness'] > 0.17 and row['tempo'] < 110:
                chill_reflective.append(row)
                print("Added to Chill Reflective Playlist")
            else:
                unselected.append(row)  # Add to the unselected playlist
                print("Added to Unselected Playlist")
        else:
            print(f"Skipping track due to missing data: {row['Track Name']} by {row['Artist']}")
    except KeyError as e:
        print(f"Missing data for track at index {index}: {e}")

# Convert to DataFrame and sort by tempo
easy_listening_df = pd.DataFrame(easy_listening).sort_values(by='tempo', ascending=True)
upbeat_dance_df = pd.DataFrame(upbeat_dance).sort_values(by='tempo', ascending=True)
chill_reflective_df = pd.DataFrame(chill_reflective).sort_values(by='tempo', ascending=True)
unselected_df = pd.DataFrame(unselected).sort_values(by='tempo', ascending=True)  # DataFrame for unselected tracks

# Save the new playlists
easy_listening_df.to_csv('Easy_Listening_Playlist.csv', index=False)
upbeat_dance_df.to_csv('Upbeat_Dance_Playlist.csv', index=False)
chill_reflective_df.to_csv('Chill_Reflective_Playlist.csv', index=False)
unselected_df.to_csv('Unselected_Playlist.csv', index=False)  # Save unselected playlist

# Print out the number of tracks in each playlist for confirmation
print(f"Easy Listening Playlist: {len(easy_listening_df)} tracks")
print(f"Upbeat Dance Playlist: {len(upbeat_dance_df)} tracks")
print(f"Chill Reflective Playlist: {len(chill_reflective_df)} tracks")
print(f"Unselected Playlist: {len(unselected_df)} tracks")


Processing Track: My Baby Just Cares for Me - 2013 Remastered Version by Nina Simone
Energy: 0.243, Acousticness: 0.829, Danceability: 0.746, Tempo: 118.578
Added to Unselected Playlist
Processing Track: I Put A Spell On You by Nina Simone
Energy: 0.398, Acousticness: 0.755, Danceability: 0.428, Tempo: 88.886
Added to Easy Listening Playlist
Processing Track: Feeling Good by Nina Simone
Energy: 0.326, Acousticness: 0.863, Danceability: 0.451, Tempo: 81.726
Added to Easy Listening Playlist
Processing Track: (You Make Me Feel Like) A Natural Woman by Aretha Franklin
Energy: 0.271, Acousticness: 0.679, Danceability: 0.603, Tempo: 110.89
Added to Unselected Playlist
Processing Track: Chain of Fools by Aretha Franklin
Energy: 0.567, Acousticness: 0.221, Danceability: 0.721, Tempo: 116.643
Added to Upbeat Dance Playlist
Processing Track: Respect - 2003 Remaster by Aretha Franklin
Energy: 0.611, Acousticness: 0.0723, Danceability: 0.744, Tempo: 114.991
Added to Upbeat Dance Playlist
Processin

In [28]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import pandas as pd

# Set your Spotify API credentials
client_id = 'efeb5f49e3494abeb5488faad890f79d'  # Replace with your client ID
client_secret = 'f115b08e67b64c9f9b3083e20456b702'  # Replace with your client secret
redirect_uri = 'http://localhost:8888/callback'  

# Define the scope for accessing and modifying playlists
scope = 'playlist-modify-public playlist-modify-private'

# Authenticate with Spotify
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=client_id,
                                                client_secret=client_secret,
                                                redirect_uri=redirect_uri,
                                                scope=scope))

# Function to create a new playlist and add tracks to it
def create_playlist_and_add_tracks(playlist_name, track_uris):
    user_id = sp.current_user()['id']
    new_playlist = sp.user_playlist_create(user_id, playlist_name)
    sp.user_playlist_add_tracks(user_id, new_playlist['id'], track_uris)
    print(f"Created and added tracks to playlist: {playlist_name}")

# Load sorted DataFrames (already sorted by tempo as per previous step)
easy_listening_df = pd.read_csv('Easy_Listening_Playlist.csv')
upbeat_dance_df = pd.read_csv('Upbeat_Dance_Playlist.csv')
chill_reflective_df = pd.read_csv('Chill_Reflective_Playlist.csv')
unselected_df = pd.read_csv('Unselected_Playlist.csv')

# Prepare track URIs from sorted DataFrames
easy_listening_uris = easy_listening_df['Spotify URI'].tolist()
upbeat_dance_uris = upbeat_dance_df['Spotify URI'].tolist()
chill_reflective_uris = chill_reflective_df['Spotify URI'].tolist()
unselected_uris = unselected_df['Spotify URI'].tolist()

# Create playlists on Spotify
create_playlist_and_add_tracks('Alyse Easy Playlist', easy_listening_uris)
create_playlist_and_add_tracks('Alyse Dance Playlist', upbeat_dance_uris)
create_playlist_and_add_tracks('Alyse Chill Reflective Playlist', chill_reflective_uris)
create_playlist_and_add_tracks('Alyse Orphans Playlist', unselected_uris)


Created and added tracks to playlist: Alyse Easy Playlist
Created and added tracks to playlist: Alyse Dance Playlist
Created and added tracks to playlist: Alyse Chill Reflective Playlist
Created and added tracks to playlist: Alyse Orphans Playlist
